## Notebook Summary: Databricks Secret Scope for Kafka Connection

**Purpose:**

This notebook securely stores and retrieves Kafka connection details using Databricks Secret Scopes. The workflow is:

1. **Define the secret scope name** (`finguard-scope`) that will hold the secrets.
2. **Extract the Databricks API URL and token** from the current notebook context — these are used to authenticate REST API calls.
3. **Create the secret scope** by calling the Databricks Secrets API (`/api/2.0/secrets/scopes/create`) with the workspace URL and bearer token.
4. **Store Kafka connection details as a secret** — bootstrap servers, topic name, API key, and API secret are bundled into a JSON string and written to the scope via the Secrets API (`/api/2.0/secrets/put`).
5. **Verify the secret** by reading it back with `dbutils.secrets.get()`, parsing the JSON, and printing the retrieved values.

> **Note:** Hardcoding the Kafka API key and secret in the notebook (Cell 7) defeats the purpose of the secret scope. In production, these credentials should be provided via interactive widgets, environment variables, or imported from an existing secret scope — never stored in plaintext within the notebook source.

In [0]:
# Secret scope name (will be created if not exists)
secret_scope_name = "finguard-scope"

In [0]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

api_url  = ctx.apiUrl().getOrElse(None)     # e.g. https://adb-...azuredatabricks.net
api_token = ctx.apiToken().getOrElse(None)  # personal access token for this session

print(api_url)
print(api_token)  # handle securely, do not log in real code

In [0]:
import requests
import json
DATABRICKS_INSTANCE = api_url  # Replace with your workspace URL
DATABRICKS_TOKEN = api_token  # Replace with your PAT

scope_name = secret_scope_name  # Scope to be created
backend_type = "DATABRICKS"     # Use "AZURE_KEYVAULT" if integrating with Key Vault


In [0]:
# ----------------------------------------
# API Endpoint to create Secret Scope
# ----------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/scopes/create"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

payload = {
    "scope": scope_name
}


# ----------------------------------------
# Send request
# ----------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# ----------------------------------------
# Handle response
# ----------------------------------------
if response.status_code == 200:
    print(f"Secret scope '{scope_name}' created successfully.")
else:
    print("Failed to create secret scope.")
    print("Status Code:", response.status_code)
    print("Response:", response.text)

In [0]:
# Fill in your real Kafka credentials below before running this cell ONCE to create the secret.
# Do not commit real values to Git — rotate them in Confluent Cloud if they ever leak.
kafka_bootstrap_servers = '<KAFKA_BOOTSTRAP_SERVERS>'  # e.g. pkc-xxxxx.<region>.gcp.confluent.cloud:9092
kafka_topic = 'my_topic'
kafka_api_key = '<KAFKA_API_KEY>'
kafka_api_secret = '<KAFKA_API_SECRET>'

kafka_connection_details = json.dumps({
    "bootstrap_servers": kafka_bootstrap_servers,
    "topic": kafka_topic,
    "api_key": kafka_api_key,
    "api_secret": kafka_api_secret
})

In [0]:
#gmail — generate an App Password in your Google account: https://myaccount.google.com/apppasswords
secret_key = 'gmail_api_key'
secret_value = '<GMAIL_APP_PASSWORD>'

#kafka
# secret_key = 'kafka_connection_details'
# secret_value = kafka_connection_details

In [0]:
# -------------------------------------------------
# API Endpoint to add our own secret to the secret scope

# -------------------------------------------------
url = f"{DATABRICKS_INSTANCE}/api/2.0/secrets/put"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

# payload = {
#     "scope": scope_name,
#     "key": 'kafka_connection_details',
#     "string_value": kafka_connection_details
# }

#gmail

payload = {
    "scope": scope_name,
    "key": secret_key,
    "string_value": secret_value
}

# -------------------------------------------------
# Send Request
# -------------------------------------------------
response = requests.post(url, headers=headers, data=json.dumps(payload))

# -------------------------------------------------
# Output
# -------------------------------------------------
if response.status_code == 200:
    print(f"Secret '{secret_value}' created successfully in scope '{scope_name}'.")
else:
    print("Failed to create secret.")
    print("Status:", response.status_code)
    print("Response:", response.text)


In [0]:
# =====================================================
# 5️⃣ Verify Secret Retrieval
# =====================================================

try:
    retrieved_json = dbutils.secrets.get(
        scope=secret_scope_name,
        key='kafka_connection_details'
    )
    
    print("Secret retrieved successfully.")
    
    parsed = json.loads(retrieved_json)
    print("Parsed JSON:")
    print(parsed)
    
except Exception as e:
    print("Secret verification failed:")
    print(str(e))
